# 06 · Discovery-aware ensembling and promotion (Experiment G, section 17)

Members enter as per-fold rank quantiles. Geographic weights and area budgets are chosen on the other folds; discovery weights are chosen with each split's withheld systems removed (`geo_masked`) or on the other splits (`loso`) - never on the labels they are scored on. An ensemble is promoted only if it passes the same section-17 gate against its best member.

Add promoted Phase-5 candidates to `members` in `configs/experiments/g1_ensemble.yaml` before running.

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'mybenkhadda/GEMS---drivendata'
COMMIT = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://github.com/{REPO}.git'   # token-free remote: a token never goes into the URL / .git/config
# private fork only: GITHUB_TOKEN (Colab Secret) is sent as an HTTP header through env vars, per command
genv = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()} if token else None
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True, env=genv)
subprocess.run(['git', '-C', '/content/GEMS', 'remote', 'set-url', 'origin', url], check=True)  # drop an old embedded token
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True, env=genv)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
colab.stage_core(); colab.stage_legacy();

In [ ]:
from gems_bench.experiment import Experiment
C = '/content/GEMS/configs/experiments/'
g = Experiment.from_yaml(C + 'g1_ensemble.yaml')
g.run_all()
display(g.rd.load_table('scoreboard'))
display(g.rd.load_table('promotion_decisions'))
display(g.rd.load_table('region_stability').pivot_table(index='region_name', columns='model', values='dti'))

In [ ]:
# Fault-recovery atlas: which withheld systems each model recovers; complementarity with G1 / gated encoder
atlas = g.rd.load_table('fault_recovery_atlas')
display(atlas.sort_values('n_models_recovering').head(20))
display(g.rd.load_table('model_complementarity'))

In [ ]:
# Model registry: promotion is explicit and needs a passing, saved decision
from gems_bench.experiment.promote import promote
from gems_bench.artifacts import ModelRegistry
reg = ModelRegistry(g.paths.registry)
display(reg.table())
# promote('ENS_weighted_rank', g.id)          # uncomment only if its decision says promote=True
# champion vs challenger figure from saved tables
import matplotlib.pyplot as plt
from gems_bench.visualization.figures import champion_selection
board = g.rd.load_table('scoreboard')
fig = champion_selection(board, board.sort_values('disc_dti_excl', ascending=False).model.iloc[0], list(board.model))
g.rd.save_fig(fig, 'champion_selection'); g.persist()